# Download Meta Global Canopy Height tiles

In [18]:
import os
import shutil

import geopandas as gpd
import requests

In [19]:
# # NCI Filepaths
# canopy_height_dir = '/scratch/xe2/cb8590/Global_Canopy_Height_v2'                                       # Initially empty directory
# canopy_height_aus_gpkg = '/home/147/cb8590/Projects/shelterbelts/tiles_global.geojson'                  # Created this by running demo_canopy_height.py, and intersecting the resulting tiles_global.py with the aus boundary in QGIS. Note: need to make sure this matches the canopy_baseurl, since v1 used 60km tiles and v2 uses 30km tiles (will get a 404 error if it doesn't match).
# tiles_gpkg = '/g/data/xe2/cb8590/Nick_Aus_treecover_10m/cb8590_Nick_Aus_treecover_10m_footprints.gpkg'  # Created this with bounding_boxes.py

In [20]:
# Local filepaths
canopy_height_dir = '/home/christopher-bradley/repos/shelterbelts/tmpdir'
canopy_height_aus_gpkg = '/home/christopher-bradley/Documents/PHD/data/Outlines/canopy_height_tiles_v2_aus.gpkg'
tiles_gpkg = '/home/christopher-bradley/repos/shelterbelts/examples/classifications/tiff_footprints_years.gpkg'

In [21]:
# canopy_baseurl = 'https://s3.amazonaws.com/dataforgood-fb-data/forests/v1/alsgedi_global_v6_float/chm/'  # v1
canopy_baseurl = 'https://s3.amazonaws.com/dataforgood-fb-data/forests/v2/global/dinov3_global_chm_v2_ml3/chm/'  # v2 (much better)

In [22]:
# Find the relevant tiles in Australia
gdf_canopy_height_aus = gpd.read_file(canopy_height_aus_gpkg)
gdf_target = gpd.read_file(tiles_gpkg).to_crs(gdf_canopy_height_aus.crs)

gdf_overlapping = (
    gpd.sjoin(
        gdf_canopy_height_aus,
        gdf_target[['geometry']],
        how='inner',
        predicate='intersects',
    )
    .drop_duplicates('tile')
)
tiles = list(gdf_overlapping['tile'])
print(f'Tiles overlapping target footprints: {len(tiles)}')

Tiles overlapping target footprints: 694


In [23]:
# Exclude any tiles we've already downloaded
to_download = [
    t for t in tiles
    if not os.path.isfile(os.path.join(canopy_height_dir, f'{t}.tif'))
]
print(f'Tiles still to download: {len(to_download)}')

Tiles still to download: 0


In [26]:
# Just download a single tile for testing
to_download = ['3112213110']

In [28]:
for tile in to_download:
    url = f'{canopy_baseurl}{tile}.tif'
    filename = os.path.join(canopy_height_dir, f'{tile}.tif')
    if requests.head(url).status_code == 200:
        with requests.get(url, stream=True) as stream, open(filename, 'wb') as out:
            shutil.copyfileobj(stream.raw, out)
        print(f'Downloaded {filename}')

print("Status code:", requests.head(url).status_code)

Downloaded /scratch/xe2/cb8590/Global_Canopy_Height_v2/3112213110.tif
Status code: 200
